В данном ноутбуке мы будем получать через скрепинг вакансии из ххру) 

In [22]:
%pip install -U selenium

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from time import sleep
import random

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.options import Options
import logging
from pathlib import Path

In [2]:
logging.basicConfig(level=logging.INFO, filename="./logs/main.log", filemode="w")

In [3]:
options = Options()
options.page_load_strategy = 'eager'
driver = webdriver.Chrome(options=options)


# Задаем нужные заголовки через девтулзы. Надо чтобы нас не засекли. 
# Скопировал из того что мой браузер шлет

custom_headers = {
    "headers": {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 YaBrowser/26.3.0.0 Safari/537.36",
        "Accept-Language": "ru,en;q=0.9",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
        "Refferer": "https://www.yandex.ru/clck/jsredir?from=yandex.ru;suggest;browser&text=",
        "sec-ch-ua-platform": "macOS"
    }
}

driver.execute_cdp_cmd('Network.setExtraHTTPHeaders', custom_headers)

{}

In [77]:
def safeExtractText(element):
    try:
        return getattr(element, 'text', '').strip()
    except:
        logging.warning(f"Не достали текст из элемента: {element.get_attribute('outerHTML')}")
        return ''

def safeGetElement(element, selector):
    try:
        return element.find_element(By.CSS_SELECTOR, selector)
    except:
        logging.warning(f"Не нашли элемент по css: {selector}")
        return None

def safeGetAndExtractText(element, selector):
    try:
        return safeExtractText(element.find_element(By.CSS_SELECTOR, selector))
    except:
        logging.warning(f"Не достали текст из элемента: {selector}")
        return ''

def safeGetAttr(element, attr):
    try:
        return element.get_attribute(attr)
    except:
        logging.warning(f"Не достали атрибут {attr} из элемента: {element.get_attribute('outerHTML')}")
        return ''

def safeXpath(element, selector):
    try:
        return element.find_element(By.XPATH, selector)
    except:
        logging.warning(f"Не нашли элемент по xpath: {selector}")
        return None

def randSleep():
    sleep(random.uniform(1, 7))


empty_df = pd.DataFrame({'title': [], 'hh_link': [], 'address': [], 'company': [], 'short_description': [], 'experience': [], 'salary': []})

In [78]:
def collect_vacancies():
    page_df = empty_df.copy()
    vacs = driver.find_elements(By.CSS_SELECTOR, "[data-qa=vacancy-serp__vacancy]")

    for vac in vacs:
        try: 
            link = safeGetAttr(safeGetElement(vac, "[data-qa=serp-item__title]"), 'href')

            duplicates = page_df.loc[page_df['hh_link'] == link]

            if len(duplicates) > 0:
                continue

            title = safeGetAndExtractText(vac, "[data-qa=serp-item__title-text]")
            address = safeGetAndExtractText(vac, "[data-qa=vacancy-serp__vacancy-address]")
            company = safeGetAndExtractText(vac, "[data-qa=vacancy-serp__vacancy-employer-text]")
            short_description_container = safeGetElement(vac, "[data-qa=vacancy-serp__vacancy_snippet_requirement]")

            short_description = ''
            if short_description_container != '':
                short_description = safeGetAndExtractText(short_description_container, "span")
            
            # Не пугайтесь это не гпт, я сам это час пытался настроить
            # Мы ищем узел с определенным data-qa, но там внутри пишут обычно частоту вылпаты
            # Поэтому поднимаемся на два уровня вверх и берем первую ноду в ней зп
            # // это любой уроверь вложенности
            xpath_salary = ".//*[starts-with(@data-qa, 'vacancy-serp__vacancy-compensation-frequency')]/../../../../*[1]"
            salary = safeExtractText(safeXpath(vac, xpath_salary))

            xpath_experience = ".//*[starts-with(@data-qa, 'vacancy-serp__vacancy-work-experience')]"
            experience = safeExtractText(safeXpath(vac, xpath_experience))

            new_row = pd.DataFrame({'title': [title], 'hh_link': [link], 'address': [address], 'company': [company], 'short_description': [short_description], 'salary': [salary], 'experience': [experience]})
            page_df = pd.concat([page_df, new_row], ignore_index=True)
        except Exception as e: 
            logging.error(f"Ошибка при обработке конкретной вакансии: {e}")
            continue
    return page_df

In [80]:
def startParsing():
    df_local = empty_df.copy()

    try: 
        logging.info(f"Открыли страницу {driver.current_url}")

        while True: 
            next = safeGetElement(driver, "[data-qa=pager-next]")
            ActionChains(driver).scroll_by_amount(0, random.randint(300, 700)).perform()

            randSleep()
            page_df = collect_vacancies()
            df_local = pd.concat([df_local, page_df], ignore_index=True)

            if next:
                ActionChains(driver).move_to_element(next).perform()
                ActionChains(driver).scroll_by_amount(0, 500).perform()

                next.click()
                next = safeGetElement(driver, "[data-qa=pager-next]")
            else:
                logging.info(f"Парсинг закончен, {len(df_local)} вакансий собрано")
                return df_local

    except Exception as e:
        logging.error(f"Ошибка при парсинге: {e}")
        logging.error(f"Текущая страница: {driver.current_url}")
        return df_local

In [81]:
# Просто с поиском ИТ
driver.get('https://hh.ru/search/vacancy?text=it&enable_snippets=true')

logging.basicConfig(level=logging.INFO, filename="./logs/hh_it.log", filemode="w", force=True)

df = startParsing()

df.to_csv('./data/hh_it.csv', index=False)

In [82]:
# Только Москва
driver.get('https://hh.ru/search/vacancy?text=it&enable_snippets=true&area=1')

logging.basicConfig(level=logging.INFO, filename="./logs/hh_moscow_it.log", filemode="w", force=True)

df = startParsing()

df.to_csv('./data/moscow_it.csv', index=False)

In [83]:
# Только Питер
driver.get('https://hh.ru/search/vacancy?text=it&enable_snippets=true&area=2')

logging.basicConfig(level=logging.INFO, filename="./logs/hh_spb_it.log", filemode="w", force=True)

df = startParsing()

df.to_csv('./data/spb_it.csv', index=False)

In [84]:
# Парсим ui ux дизайнеров
driver.get('https://hh.ru/search/vacancy?text=ui+ux+дизайнер&enable_snippets=true')

logging.basicConfig(level=logging.INFO, filename="./logs/hh_ux_it.log", filemode="w", force=True)

df = startParsing()

df.to_csv('./data/ux_it.csv', index=False)

In [85]:
# Парсим веб дизайнеров
driver.get('https://hh.ru/search/vacancy?text=веб+дизайнер&enable_snippets=true')

logging.basicConfig(level=logging.INFO, filename="./logs/hh_web_designer.log", filemode="w", force=True)

df = startParsing()

df.to_csv('./data/web_designer.csv', index=False)

In [86]:
# Парсим разрабов
driver.get('https://hh.ru/search/vacancy?text=разработчик&enable_snippets=true')

logging.basicConfig(level=logging.INFO, filename="./logs/devs.log", filemode="w", force=True)

df = startParsing()

df.to_csv('./data/devs.csv', index=False)

In [87]:
# Парсим разрабов мск
driver.get('https://hh.ru/search/vacancy?text=разработчик&enable_snippets=true&area=1')

logging.basicConfig(level=logging.INFO, filename="./logs/devs_moscow.log", filemode="w", force=True)

df = startParsing()

df.to_csv('./data/devs_moscow.csv', index=False)

In [88]:
# Парсим разрабов питер
driver.get('https://hh.ru/search/vacancy?text=разработчик&enable_snippets=true&area=2')

logging.basicConfig(level=logging.INFO, filename="./logs/devs_spb.log", filemode="w", force=True)

df = startParsing()

df.to_csv('./data/devs_spb.csv', index=False)

In [89]:
# Парсим тестировщиков
driver.get('https://hh.ru/search/vacancy?text=тестировщик&enable_snippets=true')

logging.basicConfig(level=logging.INFO, filename="./logs/testers.log", filemode="w", force=True)

df = startParsing()

df.to_csv('./data/testers.csv', index=False)

In [ ]:
# продакты 
driver.get('https://hh.ru/search/vacancy?text=project+manager&enable_snippets=true')

logging.basicConfig(level=logging.INFO, filename="./logs/product_managers.log", filemode="w", force=True)

df = startParsing()

df.to_csv('./data/product_managers.csv', index=False)

In [96]:
# аналитики  
driver.get('https://hh.ru/search/vacancy?text=аналитик+данных&enable_snippets=true')

logging.basicConfig(level=logging.INFO, filename="./logs/analysts.log", filemode="w", force=True)

df = startParsing()

df.to_csv('./data/analysts.csv', index=False)

In [3]:
# Объединяем

folder = Path("./data")

csv_files = folder.glob("*.csv")

df = pd.concat((pd.read_csv(file) for file in csv_files), ignore_index=True)

df

,title,hh_link,address,company,short_description,experience,salary
0,Разработчик,https://hh.ru/vacancy/131819443?query=%D1%80%D...,Москва,"Русская Медиагруппа, радиохолдинг",Опыт работы,Опыт 3-6 лет,NaN
1,Разработчик,https://hh.ru/vacancy/131986214?query=%D1%80%D...,Москва,Детский хоспис Дом с маяком,"Уверенное знание Python 3, базовых принципов О...",Опыт 1-3 года,"от 95 000 ₽ за месяц, на руки"
2,Backend-разработчик,https://hh.ru/vacancy/131274910?query=%D1%80%D...,Москва,ООО ГК СтиС,2+ года коммерческой разработки на PHP 7.4+. У...,Опыт 1-3 года,NaN
3,Разработчик ЦФТ,https://hh.ru/vacancy/132003073?query=%D1%80%D...,Москва,Bell Integrator,Опыт работы с АБС «ЦФТ-Банк» в качестве,Опыт 3-6 лет,NaN
4,Разработчик C++,https://hh.ru/vacancy/132001730?query=%D1%80%D...,Москва,ООО БУЛАТ,Опыт разработки или исправления/доработки внут...,Опыт 3-6 лет,NaN
...,...,...,...,...,...,...,...
13546,Бизнес-аналитик / Системный аналитик (Fullstac...,https://hh.ru/vacancy/131129287?query=%D0%BF%D...,Санкт-Петербург,ООО Апартмент системс,Опыт работы бизнес-аналитиком / системным анал...,Опыт 3-6 лет,NaN
13547,Девелопер обуви и аксессуаров,https://hh.ru/vacancy/131335836?query=%D0%BF%D...,"Москва, р-н Преображенское",Ushatava,"Аналогичный опыт работы от 3 лет, опыт работы",Опыт 1-3 года,NaN
13548,Product Analyst,https://hh.ru/vacancy/131303603?query=%D0%BF%D...,"Москва, р-н Раменки",Clear Mind,Опытом самостоятельного создания воронок и фор...,Опыт 3-6 лет,NaN
13549,Начальник отдела маркетинга,https://hh.ru/vacancy/128851476?query=%D0%BF%D...,"Челябинск, р-н Центральный",ООО Чебаркульская птица,"Опыт работы в маркетинге от 3 лет, из них не м...",Опыт 3-6 лет,NaN


In [4]:
df.duplicated().sum()

np.int64(1415)

In [5]:
df.to_csv('./df_hh.csv', index=False)